# Cross-Validation 評価ノートブック (Run All 対応)

このノートブックは、6クラスセグメンテーション（背景, conj, iris_vis, iris_occ, pupil_vis, pupil_occ）のクロスバリデーション評価を実行します。

- 入力解像度: 512x512
- モデル: U-Net (VGG16-BNエンコーダ)
- 評価指標: クラス別 Dice, 平均 Dice
- 注意: 各 fold の学習済み重み `model/method3_fold{k}_best.pth` が存在しない場合はその fold をスキップします。

実行順序（Run All推奨）
1. GPUチェック
2. 依存関係のimportと設定
3. データセット/ユーティリティ定義
4. モデル定義（UNetMethod3のみ）
5. 推論・評価ルーチン
6. GroupKFoldで評価し、表を保存


In [ ]:
# GPUチェック
import torch
print("Using device:", 'cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
がっているので

In [ ]:
# importと設定
import os, glob, math, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from sklearn.model_selection import GroupKFold
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import vgg16_bn
from torch.utils.data import Dataset, DataLoader

IMAGE_SIZE = 512
BATCH_SIZE = 8
NUM_WORKERS = 2
DATA_ROOT = Path('Images')
IMG_DIR = DATA_ROOT / 'images'
LABEL_DIR = DATA_ROOT / 'labels_seg'
MODEL_DIR = Path('model')
MODEL_DIR.mkdir(exist_ok=True)

CLASS_NAMES = ['background','conj','iris_vis','iris_occ','pupil_vis','pupil_occ']
NUM_CLASSES = 6
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# ユーティリティ: 患者ID抽出
import re

def extract_patient_id(filename: str) -> str:
    # 例: '136-20140730-9-154037_..._L_xxx.png' -> '136-20140730-9-154037'
    base = os.path.basename(filename)
    m = re.match(r"^(\d+-\d+-\d+-\d+)_", base)
    if m:
        return m.group(1)
    # フォールバック: 先頭の3フィールド結合
    parts = base.split('-')
    return '-'.join(parts[:3]) if len(parts) >= 3 else base


In [ ]:
# データセット（評価用・最小実装）
class EvalDataset(Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def _read_mask_gray(self, path):
        if not os.path.exists(path):
            return None
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        return img

    def _resize_mask(self, m):
        return cv2.resize(m, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST)

    def _resize_img(self, im):
        return cv2.resize(im, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)

    def _to_chw_tensor(self, im):
        im = im.astype(np.float32) / 255.0
        im = np.transpose(im, (2, 0, 1))
        return torch.from_numpy(im)

    def _bin_tensor(self, m):
        return torch.from_numpy((m > 127).astype(np.float32))

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self._resize_img(img)

        # ラベルファイル命名規則: 末尾に _mask_lid.png / _mask_iris.png / _mask_pupil.png
        base = os.path.splitext(os.path.basename(img_path))[0]
        lid_path   = str(LABEL_DIR / f"{base}_mask_lid.png")
        iris_path  = str(LABEL_DIR / f"{base}_mask_iris.png")
        pupil_path = str(LABEL_DIR / f"{base}_mask_pupil.png")

        lid = self._read_mask_gray(lid_path)
        iris = self._read_mask_gray(iris_path)
        pupil = self._read_mask_gray(pupil_path)

        # None対応（万一ない場合はゼロ）
        if lid is None:
            lid = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
        if iris is None:
            iris = np.zeros_like(lid)
        if pupil is None:
            pupil = np.zeros_like(lid)

        lid = self._resize_mask(lid)
        iris = self._resize_mask(iris)
        pupil = self._resize_mask(pupil)

        # 6クラス動的生成
        lid_b   = (lid   > 127)
        iris_b  = (iris  > 127)
        pupil_b = (pupil > 127)

        background = (~lid_b) & (~iris_b) & (~pupil_b)
        conj       = ( lid_b) & (~iris_b) & (~pupil_b)
        iris_vis   = ( lid_b) & ( iris_b) & (~pupil_b)
        iris_occ   = (~lid_b) & ( iris_b) & (~pupil_b)
        pupil_vis  = ( lid_b) & ( iris_b) & ( pupil_b)
        pupil_occ  = (~lid_b) & ( iris_b) & ( pupil_b)

        six = np.zeros_like(lid, dtype=np.uint8)
        six[background] = 0
        six[conj]       = 1
        six[iris_vis]   = 2
        six[iris_occ]   = 3
        six[pupil_vis]  = 4
        six[pupil_occ]  = 5

        image_t = self._to_chw_tensor(img)
        six_t   = torch.from_numpy(six.astype(np.int64))

        return {
            'image': image_t,
            'sixcls': six_t,
            'image_path': img_path
        }


In [ ]:
# UNet (VGG16-BN encoder) - Method3のみ最小実装
class UNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        v = vgg16_bn(weights='DEFAULT').features
        # VGG16-BNのブロック境界で抽出
        self.block1 = v[0:6]   # 64
        self.block2 = v[6:13]  # 128
        self.block3 = v[13:23] # 256
        self.block4 = v[23:33] # 512
        self.block5 = v[33:43] # 512
    def forward(self, x):
        x1 = self.block1(x)
        x2 = self.block2(x1)
        x3 = self.block3(x2)
        x4 = self.block4(x3)
        x5 = self.block5(x4)
        return x1, x2, x3, x4, x5

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

class UNetDecoder(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(512, 512, 2, stride=2)
        self.dec4 = ConvBlock(512+512, 256)
        self.up3 = nn.ConvTranspose2d(256, 256, 2, stride=2)
        self.dec3 = ConvBlock(256+256, 128)
        self.up2 = nn.ConvTranspose2d(128, 128, 2, stride=2)
        self.dec2 = ConvBlock(128+128, 64)
        self.up1 = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.dec1 = ConvBlock(64+64, 64)
        self.head = nn.Conv2d(64, num_classes, 1)
    def forward(self, x1, x2, x3, x4, x5):
        d4 = self.up4(x5)
        d4 = torch.cat([d4, x4], dim=1)
        d4 = self.dec4(d4)
        d3 = self.up3(d4)
        d3 = torch.cat([d3, x3], dim=1)
        d3 = self.dec3(d3)
        d2 = self.up2(d3)
        d2 = torch.cat([d2, x2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.up1(d2)
        d1 = torch.cat([d1, x1], dim=1)
        d1 = self.dec1(d1)
        out = self.head(d1)
        # VGGの入力はN,3,512,512 -> 出力も512x512
        return out

class UNetMethod3(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.encoder = UNetEncoder()
        self.decoder = UNetDecoder(num_classes)
    def forward(self, x):
        x1, x2, x3, x4, x5 = self.encoder(x)
        logits = self.decoder(x1, x2, x3, x4, x5)
        return logits


In [ ]:
# Dice関連

def one_hot(labels: torch.Tensor, num_classes: int) -> torch.Tensor:
    # labels: (N,H,W)
    n, h, w = labels.shape[0], labels.shape[1], labels.shape[2]
    out = torch.zeros((n, num_classes, h, w), dtype=torch.float32, device=labels.device)
    return out.scatter_(1, labels.unsqueeze(1), 1.0)

@torch.no_grad()
def dice_per_class(pred_logits: torch.Tensor, target_labels: torch.Tensor, eps: float=1e-6):
    # pred_logits: (N,C,H,W); target_labels: (N,H,W)
    probs = torch.softmax(pred_logits, dim=1)
    pred = torch.argmax(probs, dim=1)
    pred_oh = one_hot(pred, NUM_CLASSES)
    tgt_oh = one_hot(target_labels, NUM_CLASSES)
    dcs = []
    for c in range(NUM_CLASSES):
        p = pred_oh[:, c].reshape(pred_oh.shape[0], -1)
        t = tgt_oh[:, c].reshape(tgt_oh.shape[0], -1)
        inter = (p * t).sum(dim=1)
        denom = p.sum(dim=1) + t.sum(dim=1) + eps
        d = (2.0 * inter + eps) / denom
        dcs.append(d.mean().item())
    return dcs


In [ ]:
# 評価ルーチン
@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    all_dice = np.zeros((NUM_CLASSES,), dtype=np.float64)
    count = 0
    for batch in loader:
        images = batch['image'].to(device)
        labels = batch['sixcls'].to(device)
        logits = model(images)
        dcs = dice_per_class(logits, labels)
        all_dice += np.array(dcs)
        count += 1
    if count == 0:
        return [np.nan]*NUM_CLASSES, np.nan
    avg = (all_dice / count).tolist()
    mean_dice = float(np.nanmean(avg))
    return avg, mean_dice


In [ ]:
# GroupKFoldで評価
from tqdm import tqdm

def run_groupkfold_evaluation(n_splits=5):
    # 画像一覧
    image_paths = sorted(glob.glob(str(IMG_DIR / '*.jpg')))
    if len(image_paths) == 0:
        print('画像が見つかりません:', IMG_DIR)
        return None
    groups = [extract_patient_id(p) for p in image_paths]

    gkf = GroupKFold(n_splits=n_splits)
    records = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(image_paths, groups=groups)):
        val_images = [image_paths[i] for i in va_idx]
        val_ds = EvalDataset(val_images)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        # モデルと重み
        model = UNetMethod3(num_classes=NUM_CLASSES).to(device)
        weight_path = MODEL_DIR / f'method3_fold{fold}_best.pth'
        if weight_path.exists():
            ckpt = torch.load(weight_path, map_location=device)
            if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
                model.load_state_dict(ckpt['model_state_dict'])
            else:
                model.load_state_dict(ckpt)
            print(f"[Fold {fold}] 重み読込: {weight_path}")
        else:
            print(f"[Fold {fold}] 重みが見つかりません。スキップ: {weight_path}")
            continue

        dcs, mean_d = evaluate_model(model, val_loader, device)
        rec = { 'fold': fold, 'mean_dice': mean_d }
        for i, name in enumerate(CLASS_NAMES):
            rec[f'dice_{name}'] = dcs[i]
        records.append(rec)

    if len(records) == 0:
        print('評価できるfoldがありません（重み未検出）。')
        return None

    df = pd.DataFrame(records)
    df.loc['mean'] = df.mean(numeric_only=True)
    out_csv = MODEL_DIR / 'crossval_results_method3.csv'
    df.to_csv(out_csv, index=False)
    print('結果を保存:', out_csv)
    df


In [ ]:
# 実行
run_groupkfold_evaluation(n_splits=5)
